# Phase 0 and paired datasets

Reads what `dp2-ml phase0`, `d1-merge`, `d2-merge`, `quality` and `blends` wrote under `DP2_ML_ROOT`. Nothing is recomputed here.

1. Reference VAE (paper_baseline) on DP2 isolated galaxies: normalisation, reconstructions, domain gap.
2. D1: quality report, alignment, shapes in both surveys, orientation convention of the Euclid ellipses.
3. D2: what was kept and why the rest was rejected, examples of resolved blends.
4. Controlled blends.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dp2_cutouts.mlconfig import MLConfig
from dp2_cutouts.pairs.store import Store
from dp2_cutouts.ml import figures as F

cfg = MLConfig.load(os.environ['DP2_ML_CONFIG'])
BANDS = list(cfg['rubin']['bands'])
IB = BANDS.index(cfg['rubin']['shape_band'])
print(cfg.root)

## 1. Reference VAE on DP2 stamps

`decoder_shared` must be True: Stage 1 and Stage 2 use the same decoder. `deterministic` must be True. The normalised peak of DP2 stamps should be comparable to the training value (0.917 for a stamp at the mean training peak); a large fraction of pixels above the round-trip saturation means the unit conversion is off.

In [ ]:
p0 = cfg.eval_dir('phase0')
rep = F.load_json(p0 / 'phase0_report.json')
print('decoder shared:', rep['decoder_shared'], '| deterministic:', rep['deterministic'])
print('units:', rep['units'])
nrm = rep['normalisation']
print('scale per band:', nrm['scale_per_band'], '| training mean peak (normalised):', nrm['training_mean_peak_normalised'])
pd.DataFrame({k: v for k, v in nrm.items() if isinstance(v, dict)})

In [ ]:
pd.DataFrame(rep['domain_gap_by_snr'])

In [ ]:
ex = np.load(p0 / 'phase0_examples.npz')
F.residual_grid(ex['obs'], [ex['rec']], ['D_R(E_R(x))'], IB, ex['weight'], n=8);

In [ ]:
s0 = pd.read_parquet(p0 / 'phase0_sample.parquet')
F.scatter_e(s0, 'rec', 'obs', 'reconstruction', 'observed');

## 2. Dataset 1

In [ ]:
d1 = Store(cfg.dataset_dir('d1'))
q1 = F.load_json(d1.path / 'quality_report.json')
print(len(d1), 'pairs; overall', q1['status'])
print('rejected:', q1['rejected'])
F.report_table(q1)

In [ ]:
for k in ('astrometry', 'ivar', 'contamination', 'resolution_and_orientation', 'splits'):
    print(k, json.dumps(q1['checks'][k]['numbers'], indent=1, default=str))

In [ ]:
m = d1.meta
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
ax[0].hist(m['astrometric_offset_arcsec'], bins=100, range=(0, 0.5)); ax[0].set_xlabel('catalogue offset Euclid - Rubin [arcsec]')
ax[1].scatter(m['astrometric_west_arcsec'], m['astrometric_north_arcsec'], s=1, alpha=0.2); ax[1].set_xlim(-0.3, 0.3); ax[1].set_ylim(-0.3, 0.3)
ax[1].set_xlabel('west [arcsec]'); ax[1].set_ylabel('north [arcsec]')
ax[2].hist(m['centroid_offset_arcsec'], bins=100, range=(0, 0.5)); ax[2].set_xlabel('measured centroid offset [arcsec]')
fig.tight_layout()

Same galaxy, two frames: e1 and e2 measured on VIS and on the Rubin i stamp must correlate positively (slopes below 1 are expected: different PSF and band). The position-angle table says which convention matches the measured VIS orientation; set `euclid.pa_offset_deg` accordingly before building D2.

In [ ]:
hi = m[(m['snr_i'] > 20) & m['hsm_vis_ok'].astype(bool)].rename(columns={'e1_vis': 'vis_e1', 'e2_vis': 'vis_e2', 'e1': 'rubin_e1', 'e2': 'rubin_e2'})
F.scatter_e(hi, 'rubin', 'vis', 'Rubin i', 'Euclid VIS');
print(q1['checks']['resolution_and_orientation']['numbers'].get('position_angle'))

In [ ]:
rows = np.random.default_rng(0).choice(len(d1), 8, replace=False)
F.stamp_grid([d1.take('rubin_flux', rows), d1.take('euclid_flux', rows)], ['Rubin i', 'Euclid VIS'], n=8,
             row_labels=d1.meta['object_id'].to_numpy()[rows]);

The worst pairs of each check, to look at rather than only count:

In [ ]:
worst = [w['object_id'] for w in q1['checks']['astrometry'].get('worst', [])][:6]
rows = np.flatnonzero(d1.meta['object_id'].isin(worst).to_numpy())
if len(rows):
    F.stamp_grid([d1.take('rubin_flux', rows), d1.take('euclid_flux', rows)], ['Rubin i', 'Euclid VIS'],
                 n=len(rows), row_labels=d1.meta['object_id'].to_numpy()[rows]);

## 3. Dataset 2

In [ ]:
d2 = Store(cfg.dataset_dir('d2'))
q2 = F.load_json(d2.path / 'quality_report.json')
print(len(d2), 'targets; overall', q2['status'])
print(json.dumps(F.load_json(d2.path / 'summary.json'), indent=1))
F.report_table(q2)

In [ ]:
print(json.dumps(q2['checks']['euclid_side']['numbers'], indent=1))
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for a, c in zip(ax, ('sep_delta_e', 'sep_centroid_shift_arcsec', 'partner_sep_arcsec', 'sb_rubin')):
    a.hist(d2.meta[c].astype(float), bins=60); a.set_xlabel(c)
fig.tight_layout()

In [ ]:
rows = np.random.default_rng(1).choice(len(d2), 8, replace=False)
F.stamp_grid([d2.take('rubin_flux', rows), d2.take('euclid_flux_unmasked', rows), d2.take('euclid_flux', rows)],
             ['Rubin blend (i)', 'Euclid VIS', 'Euclid VIS, others masked'], n=8,
             row_labels=d2.meta['partner_sep_arcsec'].round(2).to_numpy()[rows]);

## 4. Controlled blends

In [ ]:
bl = Store(cfg.dataset_dir('blends'))
print(len(bl), bl.meta['split'].value_counts().to_dict())
rows = np.arange(8)
F.stamp_grid([bl.take('rubin_flux', rows), d1.take('rubin_flux', bl.meta['d1_row'].to_numpy()[rows])],
             ['blend (i)', 'target alone (i)'], n=8,
             row_labels=[f'{s:.1f}px x{q:.2f}' for s, q in zip(bl.meta['separation_px'][:8], bl.meta['flux_ratio'][:8])]);